In [ ]:
# ==========================================================
# COMPLETE CAMERA CALIBRATION USING OPENCV
# Author : Jeshwin V S
# Checkerboard : 8x8 Squares (7x7 Inner Corners)
# Square Size : 2.1 cm
# ==========================================================

import cv2
import numpy as np
import glob
import os
from google.colab.patches import cv2_imshow

# ==========================================================
# STEP 1 : UNZIP IMAGES
# ==========================================================

zip_path = "/content/checkboard.zip"

if os.path.exists(zip_path):
    !unzip -qo /content/checkboard.zip -d /content/
    print("ZIP Extracted Successfully\n")

# ==========================================================
# STEP 2 : FIND IMAGE FOLDER
# ==========================================================

image_extensions = (".jpg", ".jpeg", ".png", ".bmp")

image_folder = None

for root, dirs, files in os.walk("/content"):
    image_count = len([f for f in files if f.lower().endswith(image_extensions)])
    if image_count > 0:
        image_folder = root
        break

if image_folder is None:
    raise Exception("No calibration images found!")

print("Image Folder :", image_folder)

images = sorted(glob.glob("/content/checkboard/*"))

print("Images Found :", len(images))

# ==========================================================
# STEP 3 : SETTINGS
# ==========================================================

CHECKERBOARD = (7, 7)

SQUARE_SIZE = 2.1      # centimeters

criteria = (
    cv2.TERM_CRITERIA_EPS +
    cv2.TERM_CRITERIA_MAX_ITER,
    30,
    0.001
)

# ==========================================================
# STEP 4 : OBJECT POINTS
# ==========================================================

objp = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)

objp[:, :2] = (
    np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]]
    .T.reshape(-1, 2)
) * SQUARE_SIZE

objpoints = []
imgpoints = []

successful = 0

print("\nDetecting Checkerboard Corners...\n")

# ==========================================================
# STEP 5 : FIND CORNERS
# ==========================================================

for fname in images:

    img = cv2.imread(fname)

    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    ret, corners = cv2.findChessboardCorners(
        gray,
        CHECKERBOARD,
        cv2.CALIB_CB_ADAPTIVE_THRESH +
        cv2.CALIB_CB_NORMALIZE_IMAGE
    )

    if ret:

        successful += 1

        objpoints.append(objp)

        corners2 = cv2.cornerSubPix(
            gray,
            corners,
            (11,11),
            (-1,-1),
            criteria
        )

        imgpoints.append(corners2)

        display = img.copy()

        cv2.drawChessboardCorners(
            display,
            CHECKERBOARD,
            corners2,
            ret
        )

        print("Detected :", os.path.basename(fname))

        cv2_imshow(display)

    else:

        print("Not Detected :", os.path.basename(fname))

print("\nSuccessful Images :", successful)

# ==========================================================
# STEP 6 : CAMERA CALIBRATION
# ==========================================================

ret, cameraMatrix, distortion, rvecs, tvecs = cv2.calibrateCamera(
    objpoints,
    imgpoints,
    gray.shape[::-1],
    None,
    None
)

# ==========================================================
# STEP 7 : REPROJECTION ERROR
# ==========================================================

mean_error = 0

for i in range(len(objpoints)):

    imgpoints2, _ = cv2.projectPoints(
        objpoints[i],
        rvecs[i],
        tvecs[i],
        cameraMatrix,
        distortion
    )

    error = cv2.norm(
        imgpoints[i],
        imgpoints2,
        cv2.NORM_L2
    ) / len(imgpoints2)

    mean_error += error

mean_error /= len(objpoints)

# ==========================================================
# STEP 8 : SAVE PARAMETERS
# ==========================================================

np.savez(
    "camera_calibration.npz",
    cameraMatrix=cameraMatrix,
    distortion=distortion,
    rvecs=rvecs,
    tvecs=tvecs
)

# ==========================================================
# STEP 9 : UNDISTORT SAMPLE IMAGE
# ==========================================================

sample = cv2.imread(images[0])

h, w = sample.shape[:2]

newCameraMatrix, roi = cv2.getOptimalNewCameraMatrix(
    cameraMatrix,
    distortion,
    (w,h),
    1,
    (w,h)
)

undistorted = cv2.undistort(
    sample,
    cameraMatrix,
    distortion,
    None,
    newCameraMatrix
)

cv2.imwrite("undistorted_result.jpg", undistorted)

# ==========================================================
# STEP 10 : FIELD OF VIEW
# ==========================================================

fx = cameraMatrix[0,0]
fy = cameraMatrix[1,1]

fovx = 2 * np.degrees(np.arctan(w / (2 * fx)))
fovy = 2 * np.degrees(np.arctan(h / (2 * fy)))

# ==========================================================
# STEP 11 : PRINT RESULTS
# ==========================================================

print("\n")
print("="*70)
print("CAMERA CALIBRATION RESULTS")
print("="*70)

print("\nImage Resolution")
print("----------------")
print(f"{w} x {h}")

print("\nCheckerboard")
print("------------")
print("Squares           : 8 x 8")
print("Inner Corners     :", CHECKERBOARD)
print("Square Size       :", SQUARE_SIZE, "cm")

print("\nSuccessful Images")
print("-----------------")
print(successful)

print("\nCamera Matrix")
print("-------------")
print(cameraMatrix)

print("\nIntrinsic Parameters")
print("--------------------")
print("fx =", cameraMatrix[0,0])
print("fy =", cameraMatrix[1,1])
print("cx =", cameraMatrix[0,2])
print("cy =", cameraMatrix[1,2])

print("\nDistortion Coefficients")
print("-----------------------")
print("k1 =", distortion[0][0])
print("k2 =", distortion[0][1])
print("p1 =", distortion[0][2])
print("p2 =", distortion[0][3])
print("k3 =", distortion[0][4])

print("\nRotation Vectors")
print("----------------")
for i, r in enumerate(rvecs):
    print(f"\nImage {i+1}")
    print(r.flatten())

print("\nTranslation Vectors")
print("-------------------")
for i, t in enumerate(tvecs):
    print(f"\nImage {i+1}")
    print(t.flatten())

print("\nCalibration Quality")
print("-------------------")
print("RMS Error                :", ret)
print("Average Reprojection Err :", mean_error)

print("\nField of View")
print("-------------")
print("Horizontal FOV :", fovx)
print("Vertical FOV   :", fovy)

print("\nSaved Files")
print("-----------")
print("camera_calibration.npz")
print("undistorted_result.jpg")

print("\nOriginal Image")
cv2_imshow(sample)

print("\nUndistorted Image")
cv2_imshow(undistorted)

print("\n")
print("="*70)
print("CALIBRATION COMPLETED SUCCESSFULLY")
print("="*70)


In [ ]:
!unzip -o /content/checkboard.zip
